# Spotify Clean Playlist Maker

This notebook makes a clean (non-explicit) copy of one of your Spotify playlists. All you paste in each time is the playlist link.

## First: make your own copy
Click **File → Save a copy in Drive** and work only in your copy. Never share a copy that has your Spotify details in it.

## What it does
1. Reads every song in your playlist.
2. Keeps the songs that are already clean.
3. For each explicit song, searches Spotify for the same song by the same artist that isn't marked explicit.
4. Puts all the clean matches into a new private playlist named after the original, e.g. "My Playlist (Clean)".
5. Saves two lists you can open from the folder icon on the left:
   - `clean_songs.txt`: songs that made it into the clean playlist
   - `no_clean_version.txt`: songs with no clean version found

## One-time setup
1. Create an app at **developer.spotify.com/dashboard**. Set the Redirect URI to exactly `http://127.0.0.1:8888/callback` and tick **Web API**.
2. In the app's **Settings**, copy the **Client ID** and **Client secret**.
3. In this notebook, click the **key icon** (Secrets) on the left sidebar.
4. Add a secret named exactly `SPOTIFY_CLIENT_ID` and paste your Client ID as the value.
5. Add a second secret named exactly `SPOTIFY_CLIENT_SECRET` and paste your client secret.
6. Switch on **Notebook access** for both.

## Each time you use it
1. Run the cell below.
2. Paste your playlist link when asked (it must be a playlist **you own**; make a copy of it first if it isn't).
3. When Colab prints a link, open it and click **Agree** on Spotify.
4. You'll land on a page that says it can't be reached. That's normal. Copy the full address from the address bar, paste it into the box in Colab, and press Enter.
5. Wait for it to finish. Songs with no clean version are printed as it goes.

## Good to know
- Only songs with a clean version on Spotify can be swapped in. Some songs don't have one.
- Matches are strict (same title and same main artist), so skim `no_clean_version.txt` in case a real clean version was missed.
- Spotify limits what apps in Development Mode can do. If playlist creation is blocked, the script still saves `clean_songs.txt` so you can add the songs by hand.
- Colab forgets the login when the session resets, so you'll repeat the login step each new session.


In [ ]:
"""
Make a clean (non-explicit) copy of a Spotify playlist.
Google Colab version. You only need to paste your playlist link when asked.

ONE-TIME SETUP: add two secrets in Colab (key icon on the left sidebar):
    SPOTIFY_CLIENT_ID
    SPOTIFY_CLIENT_SECRET
and switch on "Notebook access" for both.
"""

# Install the Spotify library (this line only works in Colab)
!pip install -q spotipy

import spotipy
from spotipy.oauth2 import SpotifyOAuth
from google.colab import userdata

# ---------------- SETTINGS ----------------
REDIRECT_URI = "http://127.0.0.1:8888/callback"  # must match your Spotify app exactly
# ------------------------------------------

# Read your credentials from Colab Secrets (so they're never in the code)
try:
    CLIENT_ID = userdata.get("SPOTIFY_CLIENT_ID")
    CLIENT_SECRET = userdata.get("SPOTIFY_CLIENT_SECRET")
except Exception:
    raise SystemExit(
        "Couldn't find your secrets. Click the key icon on the left, add "
        "SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET, and switch on Notebook access."
    )

# Ask for the playlist link
PLAYLIST_URL = input("Paste your playlist link and press Enter: ").strip()

# Log in. Colab prints a link; open it, approve access, then paste back
# the address of the "can't be reached" page you land on.
sp = spotipy.Spotify(
    auth_manager=SpotifyOAuth(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        redirect_uri=REDIRECT_URI,
        scope="playlist-read-private playlist-modify-private playlist-modify-public",
        open_browser=False,  # needed for Colab
    )
)


def get_all_tracks(playlist_url):
    """Read every track in the playlist (Spotify sends them in pages of 100)."""
    tracks = []
    page = sp.playlist_items(playlist_url)
    while page:
        for entry in page["items"]:
            # Spotify renamed "track" to "item" in 2026, so check both
            track = entry.get("item") or entry.get("track")
            # Skip empty entries and local files (they have no id)
            if track and track.get("id"):
                tracks.append(track)
        page = sp.next(page) if page["next"] else None
    return tracks


def simplify(title):
    """Lowercase a title and drop extras like '(Clean)' or '- Radio Edit' so titles can be compared."""
    title = title.lower()
    if " - " in title:
        title = title.split(" - ")[0]
    if "(" in title:
        title = title.split("(")[0]
    return title.strip()


def find_clean_version(track):
    """Return the URI of a clean version of this track, or None if we can't find one."""
    # If the song is already clean, just use it
    if not track["explicit"]:
        return track["uri"]

    title = track["name"]
    main_artist = track["artists"][0]
    query = f"track:{title} artist:{main_artist['name']}"
    results = sp.search(q=query, type="track", limit=10)  # Spotify now caps this at 10

    for candidate in results["tracks"]["items"]:
        same_title = simplify(candidate["name"]) == simplify(title)
        same_artist = main_artist["id"] in [a["id"] for a in candidate["artists"]]
        if same_title and same_artist and not candidate["explicit"]:
            return candidate["uri"]

    return None


def main():
    print("Reading your playlist...")
    tracks = get_all_tracks(PLAYLIST_URL)
    print(f"Found {len(tracks)} tracks.\n")

    if not tracks:
        print("No tracks came back, so stopping here.")
        print("Make sure the playlist is one you own (make a copy if it isn't).")
        return

    # Name the new playlist after the original, e.g. "My Playlist (Clean)"
    try:
        original_name = sp.playlist(PLAYLIST_URL, fields="name")["name"]
        new_playlist_name = f"{original_name} (Clean)"
    except spotipy.SpotifyException:
        new_playlist_name = "Clean Version"

    clean_uris = []
    clean_labels = []
    missing = []

    for i, track in enumerate(tracks, start=1):
        label = f"{track['name']} - {track['artists'][0]['name']}"
        uri = find_clean_version(track)
        if uri:
            if uri not in clean_uris:  # avoid adding the same song twice
                clean_uris.append(uri)
                clean_labels.append(label)
        else:
            missing.append(label)
            print(f"[{i}/{len(tracks)}] No clean version: {label}")

    # Save both lists as text files (in Colab, see the folder icon on the left)
    with open("no_clean_version.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(missing))
    with open("clean_songs.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(clean_labels))

    print(f"\nFound clean versions for {len(clean_uris)} songs.")
    print(f"{len(missing)} songs had no clean version.")

    print(f"\nCreating playlist '{new_playlist_name}'...")
    try:
        # This is the new way to create a playlist (the old way was removed in 2026)
        new_playlist = sp.current_user_playlist_create(new_playlist_name, public=False)

        # Add songs in small batches (Spotify limits how many at a time)
        for start in range(0, len(clean_uris), 10):
            sp.playlist_add_items(new_playlist["id"], clean_uris[start:start + 10])

        print(f"Done! Added {len(clean_uris)} songs to '{new_playlist_name}'.")
    except spotipy.SpotifyException as error:
        print("Spotify wouldn't let the script create or fill the playlist:")
        print(error)
        print("\nYour clean songs were still saved to clean_songs.txt,")
        print("so you can add them to a playlist by hand.")


main()
